# Stage 2 — bounded extraction concurrency, live

Demonstrates the two things Stage 2 actually added around Stage 1's existing
text-extraction step: a **bounded concurrency cap** (S2-T03) and **SSE/DB
observability** (S2-T04) for the `ExtractionStep` node.

In production, `ExtractionStep`'s `semaphore` comes from the DI `Container`
(`Container.extraction_semaphore` in `injections/production.py`), sized from
`config/extraction.yaml`'s `max_concurrent_extractions`. This notebook builds one
directly instead, with a small cap (2) and an artificial per-call delay, so the
queueing behavior is visible in a few seconds rather than needing real concurrent
HTTP load to trigger it — same idea `tests/ingesta/test_extraction_concurrency.py`
proves automatically, just interactive here.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.

## 1 — Imports

In [1]:
import asyncio
import threading
import time

from classiflow.database.repositories.audit import InMemoryAuditRepository
from classiflow.events.broadcaster import EventBroadcaster
from classiflow.ingesta.config_extraction import get_extraction_config
from classiflow.ingesta.domain import ExtractionResult, JobContext
from classiflow.ingesta.nodes import ExtractionStep
from classiflow.services.audit.service import AuditService

_real_cap = get_extraction_config().max_concurrent_extractions
print(f"real config/extraction.yaml max_concurrent_extractions = {_real_cap}")
print("this notebook uses a smaller cap below so the queueing is visible quickly")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


real config/extraction.yaml max_concurrent_extractions = 2
this notebook uses a smaller cap below so the queueing is visible quickly


## 2 — A tracking extractor

Stands in for the real `TextExtractor` (`text_extractor` in `ExtractionStep`'s
constructor) — records how many calls were simultaneously in-flight at any point,
and sleeps briefly to simulate real extraction taking measurable time. The sleep
runs in a real OS thread (`ExtractionStep.run()` wraps the call in
`asyncio.to_thread(...)`), so it genuinely blocks that thread without blocking the
event loop — same as real MarkItDown/EasyOCR calls do.

In [2]:
class ConcurrencyTrackingExtractor:
    """Records the max number of simultaneous in-flight calls it ever saw."""

    def __init__(self, delay_seconds: float = 0.5) -> None:
        self._delay_seconds = delay_seconds
        self._lock = threading.Lock()
        self._current = 0
        self.max_observed = 0

    def __call__(self, _file_bytes: bytes, filename: str) -> ExtractionResult:
        with self._lock:
            self._current += 1
            self.max_observed = max(self.max_observed, self._current)
            print(f"  [{filename}] started  (in-flight now: {self._current})")
        time.sleep(self._delay_seconds)
        with self._lock:
            print(f"  [{filename}] finished (in-flight now: {self._current - 1})")
            self._current -= 1
        return ExtractionResult(text="x" * 30, extractor_used="tracked", char_count=30)

## 3 — Fire more jobs than the cap allows, watch them queue

6 concurrent jobs, cap = 2 — expect to see at most 2 `started` lines before the
first `finished` line, never 3+ in flight at once.

In [3]:
CAP = 2
JOB_COUNT = 6
DELAY_SECONDS = 0.5

tracker = ConcurrencyTrackingExtractor(delay_seconds=DELAY_SECONDS)
audit_repo = InMemoryAuditRepository()
audit = AuditService(audit_repo)
broadcaster = EventBroadcaster()
step = ExtractionStep(
    audit=audit,
    broadcaster=broadcaster,
    text_extractor=tracker,
    semaphore=asyncio.Semaphore(CAP),
)

t0 = time.perf_counter()
await asyncio.gather(*[
    step.run(JobContext(job_id=f"job-{i}", filename=f"doc-{i}.pdf"), b"x", f"doc-{i}.pdf")
    for i in range(JOB_COUNT)
])
elapsed = time.perf_counter() - t0

print(f"\n{JOB_COUNT} jobs through a cap of {CAP}, {DELAY_SECONDS}s each")
print(
    f"wall time: {elapsed:.2f}s  (serial would be {JOB_COUNT * DELAY_SECONDS:.2f}s, "
    f"fully parallel would be ~{DELAY_SECONDS:.2f}s)"
)
print(f"max simultaneous in-flight: {tracker.max_observed}  (cap was {CAP})")
assert tracker.max_observed <= CAP, "cap was violated!"
print("confirmed: the semaphore held the cap")

  [doc-0.pdf] started  (in-flight now: 1)
  [doc-1.pdf] started  (in-flight now: 2)


2026-08-17 16:06:29.798 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-0 node=extraction event=passed
2026-08-17 16:06:29.799 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-1 node=extraction event=passed


  [doc-0.pdf] finished (in-flight now: 1)
  [doc-1.pdf] finished (in-flight now: 0)
  [doc-2.pdf] started  (in-flight now: 1)
  [doc-3.pdf] started  (in-flight now: 2)


2026-08-17 16:06:30.306 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-2 node=extraction event=passed
2026-08-17 16:06:30.307 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-3 node=extraction event=passed


  [doc-2.pdf] finished (in-flight now: 1)
  [doc-3.pdf] finished (in-flight now: 0)
  [doc-4.pdf] started  (in-flight now: 1)
  [doc-5.pdf] started  (in-flight now: 2)


2026-08-17 16:06:30.816 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-5 node=extraction event=passed
2026-08-17 16:06:30.817 | INFO     | classiflow.services.audit.service:record:37 - audit | job=job-4 node=extraction event=passed


  [doc-5.pdf] finished (in-flight now: 1)
  [doc-4.pdf] finished (in-flight now: 0)

6 jobs through a cap of 2, 0.5s each
wall time: 1.54s  (serial would be 3.00s, fully parallel would be ~0.50s)
max simultaneous in-flight: 2  (cap was 2)
confirmed: the semaphore held the cap


## 4 — Observability: the SSE-style events `ExtractionStep` emits

Every `BaseNode` subclass (`ExtractionStep` included) emits `started`/`passed`
`NodeEvent`s through the same `EventBroadcaster` the real `/pipeline/{job_id}/events`
SSE endpoint streams from, and records duration via `AuditService` — this is S2-T04.
Re-running one job here and reading both back directly, no HTTP server needed.

In [4]:
single_job_id = "observability-demo"

events: list = []


async def collect() -> None:
    events.extend([event async for event in broadcaster.subscribe(single_job_id)])


collect_task = asyncio.create_task(collect())
await asyncio.sleep(0)  # let the subscriber attach before we emit

await step.run(JobContext(job_id=single_job_id, filename="demo.pdf"), b"x", "demo.pdf")
await broadcaster.close(single_job_id)
await collect_task

print("=== SSE events (EventBroadcaster) ===")
for e in events:
    print(f"  node={e.node:<10} status={e.status.value}")

print("\n=== audit record (AuditService / DocumentStep in production) ===")
for record in await audit_repo.list_for_job(single_job_id):
    print(f"  node={record.node:<10} event={record.event:<8} duration_ms={record.duration_ms}")

  [demo.pdf] started  (in-flight now: 1)


2026-08-17 16:06:37.933 | INFO     | classiflow.services.audit.service:record:37 - audit | job=observability-demo node=extraction event=passed


  [demo.pdf] finished (in-flight now: 0)
=== SSE events (EventBroadcaster) ===
  node=extraction status=started
  node=extraction status=passed

=== audit record (AuditService / DocumentStep in production) ===
  node=extraction event=passed   duration_ms=515
